# 06 — Regularization Techniques from Scratch

**Goal:** Implement dropout, weight decay, gradient clipping, label smoothing, and early stopping. Demonstrate their effect on overfitting.

---

## Why Regularization?

Neural networks are powerful function approximators — so powerful that they can memorize training data. Regularization adds inductive bias to prefer simpler models that generalize better.

| Technique | What it does | Key insight |
|-----------|-------------|------------|
| Dropout | Randomly zeros neurons during training | Ensemble of sub-networks |
| Weight decay | Penalizes large weights | Smoother functions, smaller effective capacity |
| Gradient clipping | Caps gradient magnitude | Prevents exploding gradients |
| Label smoothing | Softens one-hot targets | Prevents overconfident predictions |
| Early stopping | Stop before overfitting | Uses validation loss as stopping criterion |

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from matplotlib.colors import ListedColormap
np.random.seed(42)
%matplotlib inline

## Helper: Simple Neural Network

In [ ]:
class SimpleNet:
    """2-layer network for binary classification."""
    
    def __init__(self, input_dim, hidden_dim, output_dim):
        scale1 = np.sqrt(2.0 / input_dim)
        scale2 = np.sqrt(2.0 / hidden_dim)
        self.W1 = np.random.randn(input_dim, hidden_dim) * scale1
        self.b1 = np.zeros((1, hidden_dim))
        self.W2 = np.random.randn(hidden_dim, output_dim) * scale2
        self.b2 = np.zeros((1, output_dim))
    
    def forward(self, X, dropout_mask1=None):
        self.X = X
        self.z1 = X @ self.W1 + self.b1
        self.a1 = np.maximum(0, self.z1)  # ReLU
        
        if dropout_mask1 is not None:
            self.a1 = self.a1 * dropout_mask1
        self.dropout_mask1 = dropout_mask1
        
        self.z2 = self.a1 @ self.W2 + self.b2
        # Softmax
        shifted = self.z2 - np.max(self.z2, axis=1, keepdims=True)
        exp_s = np.exp(shifted)
        self.probs = exp_s / np.sum(exp_s, axis=1, keepdims=True)
        return self.probs
    
    def backward(self, y):
        N = self.X.shape[0]
        
        dz2 = self.probs.copy()
        dz2[np.arange(N), y] -= 1.0
        dz2 /= N
        
        self.dW2 = self.a1.T @ dz2
        self.db2 = np.sum(dz2, axis=0, keepdims=True)
        da1 = dz2 @ self.W2.T
        
        if self.dropout_mask1 is not None:
            da1 = da1 * self.dropout_mask1
        
        dz1 = da1 * (self.z1 > 0).astype(float)
        self.dW1 = self.X.T @ dz1
        self.db1 = np.sum(dz1, axis=0, keepdims=True)
    
    def update(self, lr, weight_decay=0.0):
        """SGD update with optional weight decay."""
        self.W1 -= lr * (self.dW1 + weight_decay * self.W1)
        self.b1 -= lr * self.db1
        self.W2 -= lr * (self.dW2 + weight_decay * self.W2)
        self.b2 -= lr * self.db2


def compute_loss(probs, y):
    N = len(y)
    return np.mean(-np.log(probs[np.arange(N), y] + 1e-12))

def compute_acc(probs, y):
    return np.mean(np.argmax(probs, axis=1) == y)

print("SimpleNet defined.")

## Create Overfitting-prone Dataset

In [ ]:
# Small noisy dataset => prone to overfitting
np.random.seed(42)
N_train, N_test = 100, 500

def make_moons(n, noise=0.3):
    t = np.linspace(0, np.pi, n // 2)
    x1 = np.c_[np.cos(t), np.sin(t)] + np.random.randn(n // 2, 2) * noise
    x2 = np.c_[np.cos(t) + 0.5, -np.sin(t) + 0.5] + np.random.randn(n // 2, 2) * noise
    X = np.vstack([x1, x2])
    y = np.hstack([np.zeros(n // 2), np.ones(n // 2)]).astype(int)
    return X, y

X_train, y_train = make_moons(N_train, noise=0.3)
X_test, y_test = make_moons(N_test, noise=0.3)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(10, 4))
for ax, X, y, title in [(ax1, X_train, y_train, f'Train (N={N_train})'),
                          (ax2, X_test, y_test, f'Test (N={N_test})')]:
    ax.scatter(X[y==0, 0], X[y==0, 1], c='#FF6B6B', s=15, alpha=0.7, label='Class 0')
    ax.scatter(X[y==1, 0], X[y==1, 1], c='#4ECDC4', s=15, alpha=0.7, label='Class 1')
    ax.set_title(title); ax.legend(); ax.set_aspect('equal')
plt.tight_layout()
plt.show()

## 1. Dropout

**Training:** Randomly zero out each neuron with probability $p$. Scale remaining by $\frac{1}{1-p}$ (inverted dropout).

**Inference:** No dropout, no scaling (inverted dropout handles this at train time).

**Why does it work?**
1. **Ensemble interpretation:** Each training step uses a different sub-network. At test time, we effectively average all $2^H$ possible sub-networks.
2. **Co-adaptation prevention:** Neurons cannot rely on specific other neurons being present => each learns more robust features.

In [ ]:
class Dropout:
    """Inverted Dropout: scale during training, no change at inference."""
    
    def __init__(self, p=0.5):
        self.p = p  # probability of DROPPING a neuron
        self.training = True
    
    def forward(self, x):
        if self.training:
            self.mask = (np.random.rand(*x.shape) > self.p).astype(float)
            self.mask /= (1.0 - self.p)  # scale up to maintain expected value
            return x * self.mask
        else:
            return x  # no change at inference
    
    def backward(self, dout):
        if self.training:
            return dout * self.mask
        return dout


# Demonstrate inverted dropout
x = np.ones((1, 10))  # all ones
drop = Dropout(p=0.5)

drop.training = True
out_train = drop.forward(x)
print(f"Training mode (p=0.5):")
print(f"  Input:  {x[0]}")
print(f"  Output: {out_train[0]}")
print(f"  Expected mean: {1.0:.1f}, actual mean: {out_train.mean():.2f}")

drop.training = False
out_test = drop.forward(x)
print(f"\nInference mode:")
print(f"  Output: {out_test[0]}  (unchanged)")

In [ ]:
# Verify: expected value is preserved
x = np.random.randn(10000, 100)
drop = Dropout(p=0.5)
outputs = [drop.forward(x.copy()) for _ in range(100)]
mean_output = np.mean(outputs, axis=0)

print(f"Input mean:           {x.mean():.4f}")
print(f"Avg output mean:      {mean_output.mean():.4f}")
print(f"Match (inverted dropout preserves E[x]): {np.allclose(x.mean(), mean_output.mean(), atol=0.05)}")

## 2. Weight Decay (L2 Regularization)

**L2 penalty in loss:** $L_{\text{total}} = L_{\text{data}} + \frac{\lambda}{2} \|W\|^2$

**Weight decay in SGD:** $W \leftarrow W - \eta(\nabla L + \lambda W) = (1 - \eta\lambda)W - \eta \nabla L$

**For SGD, these are equivalent.** But for Adam, they differ! AdamW uses decoupled weight decay:
$W \leftarrow W - \eta\lambda W - \eta \cdot \text{Adam}(\nabla L)$

In [ ]:
# Demonstrate weight decay effect
def train_model(X_train, y_train, X_test, y_test, dropout_p=0.0, weight_decay=0.0, 
                hidden=128, epochs=500, lr=0.1):
    np.random.seed(42)
    net = SimpleNet(2, hidden, 2)
    drop = Dropout(p=dropout_p)
    
    train_losses, test_losses = [], []
    train_accs, test_accs = [], []
    
    for epoch in range(epochs):
        # Training forward
        drop.training = True
        mask = None
        if dropout_p > 0:
            # Generate mask for hidden layer
            _ = drop.forward(np.ones((X_train.shape[0], hidden)))  # just to get mask
            mask = drop.mask
        
        probs = net.forward(X_train, dropout_mask1=mask)
        train_loss = compute_loss(probs, y_train)
        
        # Add L2 penalty to loss for reporting
        if weight_decay > 0:
            train_loss += 0.5 * weight_decay * (np.sum(net.W1**2) + np.sum(net.W2**2))
        
        train_losses.append(train_loss)
        train_accs.append(compute_acc(probs, y_train))
        
        # Backward
        net.backward(y_train)
        net.update(lr, weight_decay=weight_decay)
        
        # Test evaluation (no dropout)
        drop.training = False
        test_probs = net.forward(X_test)
        test_losses.append(compute_loss(test_probs, y_test))
        test_accs.append(compute_acc(test_probs, y_test))
    
    return train_losses, test_losses, train_accs, test_accs, net


print("Training function defined.")

## 3. Gradient Clipping

In [ ]:
def clip_grad_by_norm(grads, max_norm):
    """Clip gradients by global norm.
    If ||g|| > max_norm, scale g to have ||g|| = max_norm."""
    total_norm = np.sqrt(sum(np.sum(g ** 2) for g in grads))
    clip_coef = max_norm / (total_norm + 1e-6)
    if clip_coef < 1.0:
        for g in grads:
            g *= clip_coef
    return total_norm


def clip_grad_by_value(grads, clip_value):
    """Clip each gradient element to [-clip_value, clip_value]."""
    for g in grads:
        np.clip(g, -clip_value, clip_value, out=g)


# Demonstrate
g1 = np.array([3.0, 4.0])  # norm = 5
g2 = np.array([0.0, 1.0])  # norm = 1

print("Before clipping:")
print(f"  g1 = {g1}, g2 = {g2}")
print(f"  Global norm = {np.sqrt(np.sum(g1**2) + np.sum(g2**2)):.2f}")

# Clip by norm
g1_copy, g2_copy = g1.copy(), g2.copy()
norm = clip_grad_by_norm([g1_copy, g2_copy], max_norm=3.0)
print(f"\nAfter clip_by_norm(max_norm=3.0):")
print(f"  g1 = {g1_copy.round(3)}, g2 = {g2_copy.round(3)}")
print(f"  Global norm = {np.sqrt(np.sum(g1_copy**2) + np.sum(g2_copy**2)):.2f}")

# Clip by value
g1_copy2 = g1.copy()
clip_grad_by_value([g1_copy2], clip_value=2.0)
print(f"\nAfter clip_by_value(clip_value=2.0):")
print(f"  g1 = {g1_copy2}")

## 4. Label Smoothing

Instead of hard targets $[0, 0, 1, 0]$, use soft targets $[0.025, 0.025, 0.925, 0.025]$.

$$y_{\text{smooth}} = (1 - \alpha) \cdot y_{\text{onehot}} + \frac{\alpha}{C}$$

Prevents the model from becoming too confident, improves calibration.

In [ ]:
def label_smoothing(targets, num_classes, alpha=0.1):
    """Convert integer targets to smoothed one-hot vectors."""
    N = len(targets)
    one_hot = np.zeros((N, num_classes))
    one_hot[np.arange(N), targets] = 1.0
    smooth = (1.0 - alpha) * one_hot + alpha / num_classes
    return smooth


def cross_entropy_with_soft_targets(probs, soft_targets):
    """Cross-entropy with soft targets."""
    return -np.mean(np.sum(soft_targets * np.log(probs + 1e-12), axis=1))


def soft_ce_gradient(probs, soft_targets):
    """Gradient of CE with soft targets."""
    return (probs - soft_targets) / probs.shape[0]


# Demonstrate
targets = np.array([0, 1, 2])
hard = label_smoothing(targets, 3, alpha=0.0)
soft = label_smoothing(targets, 3, alpha=0.1)

print("Hard targets (alpha=0.0):")
print(hard)
print("\nSmooth targets (alpha=0.1):")
print(soft.round(4))

## 5. Early Stopping

In [ ]:
class EarlyStopping:
    """Stop training when validation loss stops improving."""
    
    def __init__(self, patience=10, min_delta=1e-4):
        self.patience = patience
        self.min_delta = min_delta
        self.best_loss = np.inf
        self.counter = 0
        self.best_weights = None
        self.stopped_epoch = None
    
    def __call__(self, val_loss, model=None):
        """Returns True if training should stop."""
        if val_loss < self.best_loss - self.min_delta:
            self.best_loss = val_loss
            self.counter = 0
            if model is not None:
                # Save best weights
                self.best_weights = {
                    'W1': model.W1.copy(), 'b1': model.b1.copy(),
                    'W2': model.W2.copy(), 'b2': model.b2.copy(),
                }
            return False
        else:
            self.counter += 1
            return self.counter >= self.patience
    
    def restore_best(self, model):
        if self.best_weights:
            model.W1 = self.best_weights['W1']
            model.b1 = self.best_weights['b1']
            model.W2 = self.best_weights['W2']
            model.b2 = self.best_weights['b2']


print("EarlyStopping defined.")

## 6. Comparison: No Reg vs Dropout vs Weight Decay

In [ ]:
configs = [
    ('No Regularization', {'dropout_p': 0.0, 'weight_decay': 0.0}),
    ('Dropout (p=0.5)', {'dropout_p': 0.5, 'weight_decay': 0.0}),
    ('Weight Decay (1e-3)', {'dropout_p': 0.0, 'weight_decay': 1e-3}),
    ('Both', {'dropout_p': 0.3, 'weight_decay': 5e-4}),
]

results = {}
for name, kwargs in configs:
    tr_l, te_l, tr_a, te_a, net = train_model(
        X_train, y_train, X_test, y_test, hidden=128, epochs=500, lr=0.1, **kwargs
    )
    results[name] = (tr_l, te_l, tr_a, te_a, net)
    print(f"{name:30s}: train_acc={tr_a[-1]:.3f}, test_acc={te_a[-1]:.3f}, "
          f"train_loss={tr_l[-1]:.4f}, test_loss={te_l[-1]:.4f}")

In [ ]:
# Plot training curves
fig, axes = plt.subplots(1, 2, figsize=(14, 5))

for name, (tr_l, te_l, tr_a, te_a, _) in results.items():
    axes[0].plot(te_l, label=name, linewidth=2)
    axes[1].plot(te_a, label=name, linewidth=2)

axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Test Loss')
axes[0].set_title('Test Loss (lower is better)')
axes[0].legend(); axes[0].grid(True, alpha=0.3)

axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Test Accuracy')
axes[1].set_title('Test Accuracy (higher is better)')
axes[1].legend(); axes[1].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## 7. Visualize Decision Boundaries

In [ ]:
def plot_decision_boundary(net, X, y, ax, title):
    h = 0.02
    x_min, x_max = X[:, 0].min() - 0.5, X[:, 0].max() + 0.5
    y_min, y_max = X[:, 1].min() - 0.5, X[:, 1].max() + 0.5
    xx, yy = np.meshgrid(np.arange(x_min, x_max, h), np.arange(y_min, y_max, h))
    grid = np.c_[xx.ravel(), yy.ravel()]
    
    probs = net.forward(grid)  # no dropout for viz
    Z = np.argmax(probs, axis=1).reshape(xx.shape)
    
    ax.contourf(xx, yy, Z, alpha=0.3, cmap=ListedColormap(['#FF6B6B', '#4ECDC4']))
    ax.scatter(X[y==0, 0], X[y==0, 1], c='#FF6B6B', s=15, edgecolors='k', linewidth=0.3)
    ax.scatter(X[y==1, 0], X[y==1, 1], c='#4ECDC4', s=15, edgecolors='k', linewidth=0.3)
    ax.set_title(title, fontsize=10)
    ax.set_aspect('equal')

fig, axes = plt.subplots(1, 4, figsize=(18, 4))

for idx, (name, (_, _, _, _, net)) in enumerate(results.items()):
    plot_decision_boundary(net, X_test, y_test, axes[idx], name)

plt.suptitle('Decision Boundaries: Effect of Regularization', y=1.02)
plt.tight_layout()
plt.show()

print("No regularization: complex, jagged boundary (overfitting).")
print("With regularization: smoother boundaries that generalize better.")

## 8. Overfitting Gap Analysis

In [ ]:
# Plot train vs test loss to show overfitting gap
fig, axes = plt.subplots(2, 2, figsize=(12, 8))

for idx, (name, (tr_l, te_l, tr_a, te_a, _)) in enumerate(results.items()):
    ax = axes[idx // 2, idx % 2]
    ax.plot(tr_l, label='Train Loss', linewidth=2)
    ax.plot(te_l, label='Test Loss', linewidth=2, linestyle='--')
    ax.fill_between(range(len(tr_l)), tr_l, te_l, alpha=0.1, color='red')
    ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
    ax.set_title(f'{name}\n(gap = overfitting)', fontsize=10)
    ax.legend(fontsize=8); ax.grid(True, alpha=0.3)

plt.suptitle('Train-Test Gap Shows Overfitting', y=1.01)
plt.tight_layout()
plt.show()

## 9. Early Stopping Demo

In [ ]:
np.random.seed(42)
net = SimpleNet(2, 128, 2)
es = EarlyStopping(patience=20)

tr_losses, te_losses = [], []
stopped = False

for epoch in range(500):
    probs = net.forward(X_train)
    tr_loss = compute_loss(probs, y_train)
    tr_losses.append(tr_loss)
    
    net.backward(y_train)
    net.update(lr=0.1)
    
    # Evaluate
    te_probs = net.forward(X_test)
    te_loss = compute_loss(te_probs, y_test)
    te_losses.append(te_loss)
    
    if not stopped and es(te_loss, net):
        es.stopped_epoch = epoch
        stopped = True
        print(f"Early stopping triggered at epoch {epoch}")
        # Continue training to show what happens

# Restore best weights
es.restore_best(net)
te_probs = net.forward(X_test)
best_acc = compute_acc(te_probs, y_test)
print(f"Best test accuracy (at early stop point): {best_acc:.3f}")

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(tr_losses, label='Train Loss')
ax.plot(te_losses, label='Test Loss')
if es.stopped_epoch:
    ax.axvline(es.stopped_epoch, color='red', linestyle='--', label=f'Early Stop (epoch {es.stopped_epoch})')
ax.set_xlabel('Epoch'); ax.set_ylabel('Loss')
ax.set_title('Early Stopping: Stop Before Overfitting Gets Worse')
ax.legend(); ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 10. Weight Decay vs L2 Regularization with Adam

For SGD: weight decay $\lambda$ is equivalent to L2 penalty $\frac{\lambda}{2}\|W\|^2$ in the loss.

For Adam: they are **NOT** equivalent! Adam's adaptive learning rate rescales gradients, so the L2 gradient ($\lambda W$) gets rescaled too. **Decoupled weight decay** (AdamW) applies decay directly to weights, not through the gradient.

In [ ]:
class Adam:
    """Adam with L2 penalty (in gradient)."""
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, l2_lambda=0.0):
        self.params = params  # list of (param, grad) tuples
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.l2_lambda = l2_lambda
        self.t = 0
        self.m = [np.zeros_like(p) for p, _ in params]
        self.v = [np.zeros_like(p) for p, _ in params]
    
    def step(self):
        self.t += 1
        for i, (param, grad) in enumerate(self.params):
            g = grad + self.l2_lambda * param  # L2 added to gradient
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g**2
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


class AdamW:
    """AdamW: decoupled weight decay (applied to weights directly)."""
    def __init__(self, params, lr=0.001, beta1=0.9, beta2=0.999, eps=1e-8, weight_decay=0.0):
        self.params = params
        self.lr = lr
        self.beta1 = beta1
        self.beta2 = beta2
        self.eps = eps
        self.weight_decay = weight_decay
        self.t = 0
        self.m = [np.zeros_like(p) for p, _ in params]
        self.v = [np.zeros_like(p) for p, _ in params]
    
    def step(self):
        self.t += 1
        for i, (param, grad) in enumerate(self.params):
            # Weight decay applied directly (not through gradient)
            param *= (1 - self.lr * self.weight_decay)
            
            g = grad  # no L2 in gradient
            self.m[i] = self.beta1 * self.m[i] + (1 - self.beta1) * g
            self.v[i] = self.beta2 * self.v[i] + (1 - self.beta2) * g**2
            m_hat = self.m[i] / (1 - self.beta1**self.t)
            v_hat = self.v[i] / (1 - self.beta2**self.t)
            param -= self.lr * m_hat / (np.sqrt(v_hat) + self.eps)


print("Adam and AdamW defined.")
print("\nKey difference:")
print("  Adam + L2:   grad = grad + lambda*W, then Adam step (L2 gets rescaled by Adam)")
print("  AdamW:       W = W*(1 - lr*lambda), then Adam step on raw grad (decay is decoupled)")

## 11. Gradient Clipping Visualization

In [ ]:
# Show gradient norms with and without clipping during training
np.random.seed(42)

def train_with_clipping(clip_norm=None, clip_value=None, lr=0.5, epochs=200):
    net = SimpleNet(2, 64, 2)
    grad_norms = []
    losses = []
    
    for epoch in range(epochs):
        probs = net.forward(X_train)
        loss = compute_loss(probs, y_train)
        losses.append(loss)
        net.backward(y_train)
        
        grads = [net.dW1, net.db1, net.dW2, net.db2]
        total_norm = np.sqrt(sum(np.sum(g**2) for g in grads))
        
        if clip_norm is not None:
            clip_grad_by_norm(grads, clip_norm)
        if clip_value is not None:
            clip_grad_by_value(grads, clip_value)
        
        clipped_norm = np.sqrt(sum(np.sum(g**2) for g in grads))
        grad_norms.append((total_norm, clipped_norm))
        
        net.update(lr)
    
    return grad_norms, losses

norms_no_clip, losses_no = train_with_clipping()
norms_clipped, losses_clip = train_with_clipping(clip_norm=1.0)

fig, (ax1, ax2) = plt.subplots(1, 2, figsize=(12, 4))

ax1.plot([n[0] for n in norms_no_clip], label='No clipping', alpha=0.7)
ax1.plot([n[1] for n in norms_clipped], label='Clipped (max_norm=1.0)', alpha=0.7)
ax1.set_xlabel('Epoch'); ax1.set_ylabel('Gradient Norm')
ax1.set_title('Gradient Norms During Training')
ax1.legend(); ax1.grid(True, alpha=0.3)

ax2.plot(losses_no, label='No clipping')
ax2.plot(losses_clip, label='With clipping')
ax2.set_xlabel('Epoch'); ax2.set_ylabel('Loss')
ax2.set_title('Training Loss')
ax2.legend(); ax2.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

## Interview Quick Reference

**Q: Dropout — why does it work? Ensemble interpretation?**  
During training, each forward pass uses a random sub-network (dropout mask). This is like training an exponential number of sub-networks with shared weights. At test time, using all neurons (with scaling) approximates the ensemble average.

**Q: Weight decay vs L2 regularization — are they the same?**  
For vanilla SGD: yes, mathematically equivalent. For Adam: NO. L2 penalty gets rescaled by Adam's adaptive learning rate (second moment). AdamW decouples weight decay from the gradient, applying it directly to weights. AdamW is preferred.

**Q: Why inverted dropout?**  
Scale activations by $1/(1-p)$ during training so that expected values match between training and inference. Alternative (non-inverted): scale by $(1-p)$ at inference. Inverted is preferred because inference code stays unchanged.

**Q: Gradient clipping — by norm vs by value?**  
By norm: preserves gradient direction, only reduces magnitude. By value: clips each element independently, can change direction. By norm is usually preferred (used in transformer training).

**Q: When to use early stopping?**  
Always monitor validation loss. Stop when it hasn't improved for `patience` epochs. Restore weights from the best epoch. It's free regularization — always use it.

---
*Next: 07_cnn_from_scratch.ipynb — convolutional neural networks*